# Wajjahni · Tourism Data Cleaning

This notebook cleans official tourism statistics for Saudi Arabia so they can be used in the Wajjahni website.

**Data:** 26 CSV files from the Saudi Tourism Authority, shared on Kaggle as
[Saudi Arabia Tourism Dataset (2015–2024)](https://www.kaggle.com/datasets/toobaik/saudi-arabia-tourism-dataset-20152024).
There is one file per region (13 regions) and per tourism type:

- **Domestic:** trips by people living in Saudi Arabia
- **Inbound:** trips by visitors from outside Saudi Arabia

**Goal:** turn the 26 messy files into two clean, tidy tables:

1. `tourism_by_region.csv`: one row per region, year, and tourism type
2. `tourism_by_destination.csv`: the same, for cities inside each region

In [1]:
import glob
import os
import pandas as pd

RAW_DIR = "data/raw/tourism"
CLEAN_DIR = "data/clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

## 1. Look at the raw data

Let's open one file and see what we're working with.

In [2]:
sample = pd.read_csv(f"{RAW_DIR}/Aseer Domestic 2024Tourism Indicators Spending csv.csv")
sample.head(12)

,DESTINATION,YEARS,Tourists Number (Overnight Visitors) (000),Overnight Stay (000),Tourists Spending (SAR Mn),Average Length of Stay (Night),Avergae Spending per Trip (SAR),Average Spending per Night (SAR)
0,Aseer Province,2015,"5,092","36,593","6,482",7.2,"1,273",177
1,Aseer Province,2016,"4,988","38,995","7,690",7.8,"1,542",197
2,Aseer Province,2017,"5,322","38,794","5,889",7.3,"1,106",152
3,Aseer Province,2018,"4,923","37,355","6,077",7.6,"1,235",163
4,Aseer Province,2019,"4,943","38,034","6,409",7.7,"1,296",169
5,Aseer Province,2020,"4,585","33,554","5,201",7.3,"1,134",155
6,Aseer Province,2021,"5,269","42,703","8,183",8.1,"1,553",192
7,Aseer Province,2022,"5,971","39,079","7,068",6.5,"1,184",181
8,Aseer Province,2023,"6,906","61,405","9,535",8.9,"1,381",155
9,Aseer Province,2024,"7,334","61,211","8,599",8.4,"1,172",140


In [3]:
print(list(sample.columns))
print()
print(sample.dtypes)

[' DESTINATION', ' YEARS ', ' Tourists Number (Overnight Visitors) (000) ', ' Overnight Stay (000) ', ' Tourists Spending (SAR Mn) ', ' Average Length of Stay (Night) ', ' Avergae Spending per Trip (SAR) ', ' Average Spending per Night (SAR) ']

DESTINATION                                        str
YEARS                                            int64
Tourists Number (Overnight Visitors) (000)         str
Overnight Stay (000)                               str
Tourists Spending (SAR Mn)                         str
Average Length of Stay (Night)                 float64
Avergae Spending per Trip (SAR)                    str
Average Spending per Night (SAR)                 int64
dtype: object


**Problems in the raw files:**

| Problem | Example | Fix |
|---|---|---|
| Column names have extra spaces, long names, and a typo | `" Avergae Spending per Trip (SAR) "` | Rename to short, clean names |
| Numbers are stored as text with commas | `"1,041"` | Remove commas and convert to numbers |
| Units are hidden in the column names | `(000)` = thousands, `(SAR Mn)` = millions | Put the unit in the new column name |
| Region totals are mixed with cities | `Aseer Province` is the total of `Abha`, `Khamis Mushait`, and `Other Aseer` | Split totals and cities into separate tables |
| Names are written inconsistently | `BILJURASHI`, `Other AlBaha` | Standardize the names |
| Region and tourism type are only in the file name | `Aseer Domestic 2024...csv` | Add them as columns |

## 2. Why not use the merged Kaggle file?

The Kaggle dataset also has a ready-made merged file, `tourism_data.csv`. Let's check its quality before using it.

In [4]:
merged = pd.read_csv("data/raw/tourism_data.csv")
print("Columns:", list(merged.columns))
print("Rows:", len(merged))
print("Exact duplicate rows:", merged.duplicated().sum())
numeric_cols = merged.columns[1:7]
print("Rows where every number is 0:", (merged[numeric_cols] == 0).all(axis=1).sum())
print()
print("Rows per province:")
print(merged["Province"].value_counts())

Columns: ['YEARS', 'Tourists_Number', 'Overnight_Stay', 'Tourists_Spending', 'Avg_Stay', 'Avg_Spending_Trip', 'Avg_Spending_Night', 'Province', 'Tourism_Type']
Rows: 1058
Exact duplicate rows: 28
Rows where every number is 0: 20

Rows per province:
Province
Makkah              118
Tabuk               111
Alqassim            100
Eastern_region      100
Madinah             100
Albaha               80
Aseer                80
Jouf                 80
Northern_borders     60
Riyadh               60
Jazan                59
Najran               58
Hail                 52
Name: count, dtype: int64


**The merged file has serious problems:**

- ❌ **The destination column was dropped**, so region totals and cities are mixed together with no way to tell them apart. Adding them up would roughly double the real numbers.
- ❌ **28 exact duplicate rows** and **20 rows that are all zeros**.

👉 So instead, we'll **rebuild a clean dataset from the 26 original files**, which still have the destination column.

## 3. Load all 26 files

We read every file, use `thousands=","` so `"1,041"` becomes `1041`, and add the **region** and **tourism type** from the file name.

We also rename the regions to match the names used on the Wajjahni website.

In [5]:
REGION_NAMES = {
    "Albaha": "Al-Baha",
    "Alqassim": "Al-Qassim",
    "Aseer": "Aseer",
    "Eastern": "Eastern Province",
    "Hail": "Hail",
    "Jazan": "Jazan",
    "Jouf": "Al-Jouf",
    "Madinah": "Madinah",
    "Makkah": "Makkah",
    "Najran": "Najran",
    "Northern": "Northern Borders",
    "Riyadh": "Riyadh",
    "Tabuk": "Tabuk",
}

NEW_COLUMNS = [
    "destination",
    "year",
    "tourists_thousands",
    "overnight_stays_thousands",
    "spending_sar_millions",
    "avg_stay_nights",
    "avg_spending_per_trip_sar",
    "avg_spending_per_night_sar",
]

frames = []
for path in sorted(glob.glob(f"{RAW_DIR}/*.csv")):
    file_name = os.path.basename(path)
    region_key = file_name.split()[0]
    tourism_type = "Domestic" if "Domestic" in file_name else "Inbound"

    df = pd.read_csv(path, thousands=",", encoding="utf-8-sig")
    df.columns = NEW_COLUMNS
    df["region"] = REGION_NAMES[region_key]
    df["tourism_type"] = tourism_type
    frames.append(df)

tourism = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(frames)} files, {len(tourism)} rows")
tourism.head()

Loaded 26 files, 1058 rows


,destination,year,tourists_thousands,overnight_stays_thousands,spending_sar_millions,avg_stay_nights,avg_spending_per_trip_sar,avg_spending_per_night_sar,region,tourism_type
0,Al Bahah Province,2015,1041.0,8807.0,1120.0,8.5,1075.0,127.0,Al-Baha,Domestic
1,Al Bahah Province,2016,1223.0,9707.0,1350.0,7.9,1104.0,139.0,Al-Baha,Domestic
2,Al Bahah Province,2017,1106.0,7126.0,1028.0,6.4,929.0,144.0,Al-Baha,Domestic
3,Al Bahah Province,2018,1235.0,8864.0,1380.0,7.2,1118.0,156.0,Al-Baha,Domestic
4,Al Bahah Province,2019,1462.0,11016.0,1666.0,7.5,1140.0,151.0,Al-Baha,Domestic


In [6]:
tourism.dtypes

destination                       str
year                            int64
tourists_thousands            float64
overnight_stays_thousands     float64
spending_sar_millions         float64
avg_stay_nights               float64
avg_spending_per_trip_sar     float64
avg_spending_per_night_sar    float64
region                            str
tourism_type                      str
dtype: object

All the number columns are now real numbers (`int64` or `float64`) instead of text. ✅

## 4. Clean the destination names

In [7]:
tourism["destination"] = tourism["destination"].str.strip()
sorted(tourism["destination"].unique())

['Abha',
 'Al Ahsa',
 'Al Bahah',
 'Al Bahah Province',
 'Al Khobar',
 'Al Qurayyat',
 'Al Ula',
 'Al Wajh',
 'Alqassim Province',
 'Arar',
 'Aseer Province',
 'BILJURASHI',
 'Buraydah',
 'Dammam',
 'Duba',
 'Eastern Province',
 'Hail',
 'Hail Province',
 'Jazan',
 'Jazan Province',
 'Jeddah',
 'Jouf Province',
 'Khamis Mushait',
 'Madinah',
 'Madinah Province',
 'Makkah',
 'Makkah Province',
 'Najran',
 'Najran Province',
 'Northern Borders Province',
 'Other AlBaha',
 'Other AlQassim',
 'Other Aseer',
 'Other Eastern',
 'Other Hail',
 'Other Jazan',
 'Other Jouf',
 'Other Madinah',
 'Other Makkah',
 'Other Najran',
 'Other Northern Borders',
 'Other Riyadh',
 'Other Tabuk',
 'Rabigh',
 'Rass',
 'Riyadh',
 'Riyadh Province',
 'Sakaka',
 'Tabuk',
 'Tabuk Province',
 'Taif',
 'Umluj',
 'Unayzah',
 'Yanbu']

Each region has:
- a **total row**, whose name ends with `Province` (for example `Aseer Province`)
- **city rows** (for example `Abha`)
- an **"Other …" row** for the rest of the region

We mark which rows are totals, and clean up names like `BILJURASHI` → `Biljurashi` and `Other AlBaha` → `Other Al-Baha`.

In [8]:
tourism["is_region_total"] = tourism["destination"].str.endswith("Province")

DESTINATION_FIXES = {
    "BILJURASHI": "Biljurashi",
    "Other AlBaha": "Other Al-Baha",
    "Other AlQassim": "Other Al-Qassim",
    "Al Ula": "AlUla",
}
tourism["destination"] = tourism["destination"].replace(DESTINATION_FIXES)

# For "Other ..." rows, use the website's region name, e.g. "Other Eastern" -> "Other Eastern Province"
is_other = tourism["destination"].str.startswith("Other ")
tourism.loc[is_other, "destination"] = "Other " + tourism.loc[is_other, "region"]

sorted(tourism.loc[~tourism["is_region_total"], "destination"].unique())

['Abha',
 'Al Ahsa',
 'Al Bahah',
 'Al Khobar',
 'Al Qurayyat',
 'Al Wajh',
 'AlUla',
 'Arar',
 'Biljurashi',
 'Buraydah',
 'Dammam',
 'Duba',
 'Hail',
 'Jazan',
 'Jeddah',
 'Khamis Mushait',
 'Madinah',
 'Makkah',
 'Najran',
 'Other Al-Baha',
 'Other Al-Jouf',
 'Other Al-Qassim',
 'Other Aseer',
 'Other Eastern Province',
 'Other Hail',
 'Other Jazan',
 'Other Madinah',
 'Other Makkah',
 'Other Najran',
 'Other Northern Borders',
 'Other Riyadh',
 'Other Tabuk',
 'Rabigh',
 'Rass',
 'Riyadh',
 'Sakaka',
 'Tabuk',
 'Taif',
 'Umluj',
 'Unayzah',
 'Yanbu']

## 5. Check for missing values, duplicates, and zeros

In [9]:
print("Duplicate rows:", tourism.duplicated().sum())
print()
print("Missing values per column:")
print(tourism.isna().sum())

Duplicate rows: 0

Missing values per column:
destination                    0
year                           0
tourists_thousands            22
overnight_stays_thousands     12
spending_sar_millions         18
avg_stay_nights                2
avg_spending_per_trip_sar      2
avg_spending_per_night_sar     2
region                         0
tourism_type                   0
is_region_total                0
dtype: int64


In [10]:
missing = tourism[tourism.isna().any(axis=1)]
print("Rows with missing values by tourism type:")
print(missing["tourism_type"].value_counts())
missing.head(8)

Rows with missing values by tourism type:
tourism_type
Inbound    28
Name: count, dtype: int64


,destination,year,tourists_thousands,overnight_stays_thousands,spending_sar_millions,avg_stay_nights,avg_spending_per_trip_sar,avg_spending_per_night_sar,region,tourism_type,is_region_total
161,Rass,2016,0.2,NaN,NaN,2.0,2092.0,1046.0,Al-Qassim,Inbound,False
232,Abha,2017,NaN,NaN,NaN,NaN,NaN,NaN,Aseer,Inbound,False
241,Khamis Mushait,2016,NaN,NaN,NaN,3.0,2950.0,983.0,Aseer,Inbound,False
242,Khamis Mushait,2017,NaN,NaN,NaN,NaN,NaN,NaN,Aseer,Inbound,False
244,Khamis Mushait,2019,NaN,NaN,NaN,7.7,3035.0,393.0,Aseer,Inbound,False
245,Khamis Mushait,2020,NaN,NaN,NaN,3.0,1900.0,633.0,Aseer,Inbound,False
411,Other Hail,2016,1.0,2.0,NaN,3.0,338.0,113.0,Hail,Inbound,False
443,Jazan Province,2016,NaN,1.0,1.0,3.0,3968.0,1323.0,Jazan,Inbound,True


**About the missing values:** they appear almost only in **inbound** data for **small destinations** (for example `Rass` or `Other Jazan`), where the numbers are very small. The source leaves them empty instead of publishing tiny numbers.

👉 We **keep them as missing (`NaN`)** instead of filling them with 0, because 0 would mean "no tourists at all", which isn't what the source says.

## 6. Find rows that ended up in the wrong file

Each destination should have **one row per year**. Let's check for repeats.

In [11]:
key = ["region", "tourism_type", "destination", "year"]
repeats = tourism[tourism.duplicated(key, keep=False)]
repeats

,destination,year,tourists_thousands,overnight_stays_thousands,spending_sar_millions,avg_stay_nights,avg_spending_per_trip_sar,avg_spending_per_night_sar,region,tourism_type,is_region_total
807,Najran Province,2024,24.0,1051.0,56.0,43.2,2309.0,53.0,Najran,Inbound,True
808,Najran Province,2024,1259.0,11352.0,2115.0,9.0,1680.0,186.0,Najran,Inbound,True


`Najran Province` has **two rows for 2024 in the inbound file**. Let's compare them with the **domestic** file:

In [12]:
tourism[(tourism["destination"] == "Najran Province") & (tourism["year"] == 2024)]

,destination,year,tourists_thousands,overnight_stays_thousands,spending_sar_millions,avg_stay_nights,avg_spending_per_trip_sar,avg_spending_per_night_sar,region,tourism_type,is_region_total
778,Najran Province,2024,1259.0,11352.0,2115.0,9.0,1680.0,186.0,Najran,Domestic,True
807,Najran Province,2024,24.0,1051.0,56.0,43.2,2309.0,53.0,Najran,Inbound,True
808,Najran Province,2024,1259.0,11352.0,2115.0,9.0,1680.0,186.0,Najran,Inbound,True


The second inbound row (1,259 thousand tourists) is **exactly the same as the domestic row**. It was copied into the inbound file by mistake. The real inbound number is the first one (24 thousand), which fits the earlier years (7–17 thousand).

👉 We remove any inbound row that is an exact copy of the domestic row for the same destination and year.

In [13]:
value_cols = NEW_COLUMNS[1:]  # year and all the numbers
domestic_rows = tourism[tourism["tourism_type"] == "Domestic"][["destination"] + value_cols]

inbound = tourism[tourism["tourism_type"] == "Inbound"].reset_index()
copied = inbound.merge(domestic_rows, on=["destination"] + value_cols, how="inner")["index"]

print("Rows copied from the domestic file by mistake:", len(copied))
tourism = tourism.drop(index=copied).reset_index(drop=True)
print("Repeats left:", tourism.duplicated(key).sum())

Rows copied from the domestic file by mistake: 1
Repeats left: 0


Let's also check that every region has all 10 years (2015–2024):

In [14]:
years = tourism[tourism["is_region_total"]].groupby(["region", "tourism_type"])["year"].nunique()
years[years < 10]

region  tourism_type
Najran  Inbound         9
Name: year, dtype: int64

A few region totals are missing a year in the source (for example, Najran inbound has no 2017 row). We leave these gaps as they are and keep them in mind when making charts.

## 7. Validate: do the cities add up to the region total?

If the data is consistent, the cities in a region should add up to roughly the region total (small differences come from rounding).

In [15]:
totals = tourism[tourism["is_region_total"]].set_index(["region", "tourism_type", "year"])["tourists_thousands"]
city_sums = (
    tourism[~tourism["is_region_total"]]
    .groupby(["region", "tourism_type", "year"])["tourists_thousands"]
    .sum()
)

check = pd.DataFrame({"region_total": totals, "sum_of_cities": city_sums})
check["difference_pct"] = ((check["sum_of_cities"] - check["region_total"]) / check["region_total"] * 100).round(1)

print("Rows checked:", len(check))
print("Within 2% of the total:", (check["difference_pct"].abs() <= 2).sum())
check.sort_values("difference_pct", key=abs, ascending=False).head(8)

Rows checked: 259
Within 2% of the total: 242


,,,region_total,sum_of_cities,difference_pct
region,tourism_type,year,,,
Najran,Inbound,2016,1.0,1.7,70.0
Al-Baha,Inbound,2020,2.0,1.0,-50.0
Jazan,Inbound,2021,5.0,4.0,-20.0
Najran,Inbound,2019,7.0,6.0,-14.3
Jazan,Inbound,2020,9.0,8.0,-11.1
Al-Baha,Inbound,2021,1.0,1.1,10.0
Jazan,Inbound,2018,13.0,12.0,-7.7
Najran,Inbound,2023,17.0,18.0,5.9


Most cities add up to their region total, which confirms the data is consistent. The few bigger differences are in small inbound numbers, where rounding and the missing values above matter more.

## 8. Split into two clean tables and save

In [16]:
COLUMN_ORDER = [
    "region", "year", "tourism_type",
    "tourists_thousands", "overnight_stays_thousands", "spending_sar_millions",
    "avg_stay_nights", "avg_spending_per_trip_sar", "avg_spending_per_night_sar",
]

by_region = (
    tourism[tourism["is_region_total"]][COLUMN_ORDER]
    .sort_values(["region", "tourism_type", "year"])
    .reset_index(drop=True)
)

by_destination = (
    tourism[~tourism["is_region_total"]][["region", "destination"] + COLUMN_ORDER[1:]]
    .sort_values(["region", "destination", "tourism_type", "year"])
    .reset_index(drop=True)
)

by_region.to_csv(f"{CLEAN_DIR}/tourism_by_region.csv", index=False)
by_destination.to_csv(f"{CLEAN_DIR}/tourism_by_destination.csv", index=False)

print("tourism_by_region.csv:", by_region.shape)
print("tourism_by_destination.csv:", by_destination.shape)
by_region.head()

tourism_by_region.csv: (259, 9)
tourism_by_destination.csv: (798, 10)


,region,year,tourism_type,tourists_thousands,overnight_stays_thousands,spending_sar_millions,avg_stay_nights,avg_spending_per_trip_sar,avg_spending_per_night_sar
0,Al-Baha,2015,Domestic,1041.0,8807.0,1120.0,8.5,1075.0,127.0
1,Al-Baha,2016,Domestic,1223.0,9707.0,1350.0,7.9,1104.0,139.0
2,Al-Baha,2017,Domestic,1106.0,7126.0,1028.0,6.4,929.0,144.0
3,Al-Baha,2018,Domestic,1235.0,8864.0,1380.0,7.2,1118.0,156.0
4,Al-Baha,2019,Domestic,1462.0,11016.0,1666.0,7.5,1140.0,151.0


## 9. Quick look at the clean data

Which regions had the most tourists in 2024?

In [17]:
latest = by_region[by_region["year"] == 2024]
(
    latest.pivot_table(index="region", columns="tourism_type", values="tourists_thousands", aggfunc="sum")
    .assign(Total=lambda d: d.sum(axis=1))
    .sort_values("Total", ascending=False)
)

tourism_type,Domestic,Inbound,Total
region,,,
Makkah,23487.0,19570.0,43057.0
Riyadh,17597.0,2861.0,20458.0
Eastern Province,13644.0,4998.0,18642.0
Aseer,7334.0,158.0,7492.0
Madinah,6721.0,439.0,7160.0
Al-Qassim,4872.0,317.0,5189.0
Tabuk,3090.0,154.0,3244.0
Jazan,2936.0,13.0,2949.0
Hail,1841.0,326.0,2167.0


## Summary

- Loaded **26 official files** (13 regions × domestic and inbound, 2015–2024)
- Found that the merged Kaggle file is unreliable (no destination column, duplicates, zero rows), so rebuilt the data from the original files
- Renamed columns, converted text numbers to real numbers, and made the units clear
- Found and removed a row that was copied into the wrong file (Najran inbound 2024)
- Matched region names to the Wajjahni website
- Separated region totals from cities, and checked that they add up
- Kept missing values as missing instead of guessing

**Output files** in `data/clean/`:
- `tourism_by_region.csv`
- `tourism_by_destination.csv`

**Next step:** use these files to build a data analysis page on the Wajjahni website.